In [1]:
import os
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import pandas as pd
import json
from pathlib import Path
from diffusers.utils import load_image

# os.environ["CUDA_VISIBLE_DEVICES"] = "0" # use GPU 0 for diffusion model and GPU 1 for Qwen3-VL 

In [ ]:
import torch
from diffusers import DiffusionPipeline, FlowMatchEulerDiscreteScheduler   

# 1. load Qwen-Image-Edit base model (same source as the .safetensors in ComfyUI)
# Replace "Qwen/Qwen-Image-Edit-2509" with your actual HuggingFace repo
pipe = DiffusionPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit-2509",     # repo id = Qwen/Qwen-Image-Edit-2509
    torch_dtype=torch.bfloat16,       
)

# 2. replace scheduler 
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(pipe.scheduler.config)

# 3.（可选）替换自定义 VAE —— 对应你在 ComfyUI 里选的那个 VAE 节点
# 如果你在 ComfyUI 用的是单独 VAE safetensors，这里可以手动加载：
# from diffusers import AutoencoderKL
# vae = AutoencoderKL.from_single_file("/home/zzou/ComfyUI/models/vae/qwen_image_vae.safetensors", torch_dtype=torch.float16)
# pipe.vae = vae

# 4. add Lightning LoRA - ComfyUI 's “Lightning LoRA” noede
pipe.load_lora_weights(
    "ComfyUI/models/loras/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors",       
    adapter_name="lightning"
)
pipe.set_adapters(["lightning"], [1.0])  # LoRA =1.0

pipe.enable_lora()
pipe.enable_sequential_cpu_offload() # save GPU memory very much, use only 1GB VRAM
# pipe.enable_model_cpu_offload() # using this one, when moving to GPU still gets OOM
# pipe.to("cuda") # get OOM directly if using this line

# 6. “4 steps lightning ” -> 8 steps
def edit_image_lightning(pipe, image, prompt, seed=42):
    """
    correspond to ComfyUI Qwen2.5-VL encoding + Qwen-Image-Edit-2509 UNet + 
    Lightning LoRA + KSampler(Euler, steps=4)
    """
    generator = torch.Generator(device="cuda").manual_seed(seed)

    out = pipe(
        prompt=prompt,          
        image=image,            
        num_inference_steps=4,  #  KSampler steps = 4 (Lightning)
        true_cfg_scale=3.0,  # Lightning LoRA default
        generator=generator,
        negative_prompt=" ",
    )
    return out.images[0]


# 7. 
def edit_image_20steps(pipe, image, prompt, seed=42):
    """
    correspond to ComfyUI: workflow KSampler steps=20。
    """
    generator = torch.Generator(device="cuda").manual_seed(seed)

    out = pipe(
        prompt=prompt,
        image=image,
        num_inference_steps=20,  # 对应 KSampler steps = 20
        guidance_scale=5.0,      
        generator=generator,
    )
    return out.images[0]


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

### get invalid image paths 

In [3]:
version0_dataset_path_chair = "./Version0_dataset/Qwen_Image_Edit_2509_v0"
version0_dataset_path_table = "./Version0_dataset/Qwen_Image_Edit_2509_v0_table"
version0_dataset_path_armchair = "./Version0_dataset/Qwen_Image_Edit_2509_v0_armchair"

valid_image_paths = json.load(open("./valide_image_paths.json"))
valid_set = set(valid_image_paths)

folders = [
    version0_dataset_path_chair,
    version0_dataset_path_table,
    version0_dataset_path_armchair
]

invalid_paths = []

for folder in folders:
    for root, _, files in os.walk(folder):
        for f in files:
            if not f.lower().endswith(".png"):
                continue
            path = os.path.join(root, f)
            if not path.startswith("./"):
                path = "./" + path
            if path not in valid_set:
                invalid_paths.append(path)
print(f"Total invalid images: {len(invalid_paths)}")

# original_images_paths = []
# for path in invalid_paths:
#     filename = os.path.basename(path)
#     prefix = filename.split("FACE")[0].rstrip("_")  # 获取 "FACE" 前的部分并去掉末尾的下划
#     original_image = "./WhatsUp_dataset/controlled_images/" +  prefix + ".jpeg"
#     original_images_paths.append(original_image)

Total invalid images: 67


### load qwen3-vl and ask validation questions

In [4]:
model_id = "Qwen/Qwen3-VL-8B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype="auto",
    device_map={"": "cuda:1"},
    # device_map="auto",
)

model.eval()

def ask_yes_no(img_path, question):
    prompt = f"{question} Answer with exactly one word: yes or no."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": "file://" + os.path.abspath(img_path)},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    images, videos = process_vision_info(messages, image_patch_size=16)

    inputs = processor(text=[text], images=images, videos=videos, do_resize=False, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    # print("pixel_values:", inputs["pixel_values"].shape, inputs["pixel_values"].dtype) # pixel_values: torch.Size([4144, 1536]) torch.float32
    # if "image_grid_thw" in inputs:
    #     print("image_grid_thw:", inputs["image_grid_thw"].shape, inputs["image_grid_thw"][:2]) # image_grid_thw: torch.Size([1, 3]) tensor([[ 1, 56, 74]], device='cuda:0')


    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
        )
        # do_sample=False, ##. ## try first generate sth correct, then try generate logis\ts / probs
        # use_cache=True,
    

    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], output)]
    response = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return response.strip().lower()


def check_image_validity(img_path, second_object, prefix=None, postfix=None):
    second_obj_direction = prefix.split("of")[0].rstrip("_").split("_")[-1] if prefix else ""
    questions_template = [
        "Is there only 1 human in this image?",
        "Is the {second_object} next to the human?",
        "Is the {second_object} in the image?",
        "Is the {second_object} clearly recognisable?",
        "Is the {second_object} {second_obj_direction} of the human?",
        "Is the human facing {postfix}?", 
    ]
    answers = []
    for q_template in questions_template:
        question = q_template.format(second_object=second_object)
        answer = ask_yes_no(img_path, question)
        answers.append(answer)

        if answer != "yes":
            print(f"{question} not yes: {img_path}")
            return False
    return True


use_kernel_func_from_hub is not available in the installed kernels version. Please upgrade kernels to use this feature.


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

In [ ]:
# def check_image_validity(img_path, second_object):
#     questions_template = [
#         "Is there only 1 human in this image?",
#         "Is the {second_object} next to the human?",
#         "Is the {second_object} in the image?",
#         "Is the {second_object} clearly recognisable?",
#     ]
#     answers = []
#     for q_template in questions_template:
#         question = q_template.format(second_object=second_object)
#         answer = ask_yes_no(img_path, question)
#         answers.append(answer)

#         if answer != "yes":
#             print(f"{question} not yes: {img_path}")
#             return False
#     return True

In [9]:
# EDIT CHAIR IMAGES
import random
from diffusers.utils import load_image

folder_chair = version0_dataset_path_chair 
EDIT_PROMPT_FACE_CAMERA_BASE  = ("Insert a human sitting on the chair facing the camera, with their body also faced the camera. Do not remove the {second_object}. Preserve the scene layout and keep the {second_object} visible.")
EDIT_PROMPT_FACE_LEFT_BASE  = ("Insert a human sitting on the chair facing left, with their body also turned to the left. Do not remove the {second_object}. Preserve the scene layout and keep the {second_object} visible.")
EDIT_PROMPT_FACE_RIGHT_BASE  = ("Insert a human sitting on the chair facing right, with their body also turned to the right. Do not remove the {second_object}. Preserve the scene layout and keep the {second_object} visible.")
dst_dir = Path("./Version0_dataset/Qwen_Image_Edit_2509_v0_chair_2nd")
dst_dir.mkdir(parents=True, exist_ok=True)

prompt_map = {
    "CAMERA": EDIT_PROMPT_FACE_CAMERA_BASE,
    "LEFT": EDIT_PROMPT_FACE_LEFT_BASE,
    "RIGHT": EDIT_PROMPT_FACE_RIGHT_BASE,
}

# max_retry = 10

for root, _, files in os.walk(folder_chair):
    for f in files:
        if not f.lower().endswith(".png"):
            continue
        path = os.path.join(root, f)
        if not path.startswith("./"):
            path = "./" + path
        if path not in valid_set:
            flag = False # reset flag for each new image
            filename = os.path.basename(path)
            prefix = filename.split("FACE")[0].rstrip("_")  # 获取 "FACE" 前的部分并去掉末尾的下划
            postfix = filename.split("FACE-")[1].split(".")[0]  # 获取 "FACE" 后的部分（不包括扩展名）
            # print(f"Processing {path}, prefix: {prefix}, postfix: {postfix}")
            original_image = "./WhatsUp_dataset/controlled_images/" +  prefix + ".jpeg"
            # print(original_image)
            second_object = prefix.split("_", 1)[0]
            # edit face camera

            # for attempt in range(max_retry):
            attempt = 0
            img = load_image(original_image).convert("RGB")
            while not flag:
                with torch.inference_mode():
                    out_img = edit_image_lightning(
                        pipe,
                        image=img,
                        prompt=prompt_map[postfix].format(second_object=second_object),
                        seed = random.randint(0, 10**9), 
                    )
                    out_path = dst_dir / f"{prefix}_FACE-{postfix}.png"
                    out_img.save(out_path)
                flag = check_image_validity(str(out_path), second_object, prefix, postfix)
                attempt += 1
                print(f"Attempt {attempt} for {original_image}, validity: {flag}")
            #     if flag:
            #         break
            # if not flag:
            #     print(f"Failed after {max_retry} tries: {original_image}")


true_cfg_scale is passed as 3.0, but classifier-free guidance is not enabled since no negative_prompt is provided.


  0%|          | 0/4 [00:00<?, ?it/s]

true_cfg_scale is passed as 3.0, but classifier-free guidance is not enabled since no negative_prompt is provided.


Is the mug next to the human? not yes: Version0_dataset/Qwen_Image_Edit_2509_v0_chair_2nd/mug_left_of_chair_FACE-LEFT.png
Attempt 1 for ./WhatsUp_dataset/controlled_images/mug_left_of_chair.jpeg, validity: False


  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

true_cfg_scale is passed as 3.0, but classifier-free guidance is not enabled since no negative_prompt is provided.


Is the mug next to the human? not yes: Version0_dataset/Qwen_Image_Edit_2509_v0_chair_2nd/mug_left_of_chair_FACE-LEFT.png
Attempt 3 for ./WhatsUp_dataset/controlled_images/mug_left_of_chair.jpeg, validity: False


  0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 